# 🧵 Raw String Comparator

**Purpose:** 3 column wali CSV do — script compare karke batayegi kaunsi rows match hain aur kaunsi nahi.

| Column | Source |
|--------|--------|
| `Notebook Rawstring` | Source file (notepad) se copy ki hui strings |
| `Automation Script Rawstring` | Script ki banai hui raw strings |
| `Trino Rawstring` | Trino se fetch ki hui `raw_str` values |

### Checks
| Check | Description |
|-------|-------------|
| NB == AUTO | Notepad == Automation script |
| NB == TRINO | Notepad == Trino |
| AUTO == TRINO | Automation == Trino |
| ALL SAME | Teeno identical |


---
## 📦 Cell 1 — Libraries

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 200)
print('✅ Libraries loaded!')

---
## ⚙️ Cell 2 — Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CSV FILE PATH — 3 columns wali file
# ─────────────────────────────────────────────────────────────────
INPUT_CSV = r"/home/shariq/Downloads/rawstrings.csv"

# Column names — agar tumhare CSV mein alag names hain toh yahan update karo
COL_NOTEBOOK   = 'Notebook Rawstring'
COL_AUTOMATION = 'Automation Script Rawstring'
COL_TRINO      = 'Trino Rawstring'

print(f'✅ Input CSV : {INPUT_CSV}')
print(f'✅ Columns   : [{COL_NOTEBOOK}] | [{COL_AUTOMATION}] | [{COL_TRINO}]')

---
## 📂 Cell 3 — CSV Load + Compare

In [ ]:
# ── Load ──
df = pd.read_csv(INPUT_CSV)
df.columns = df.columns.str.strip()

# Validate columns exist
for col in [COL_NOTEBOOK, COL_AUTOMATION, COL_TRINO]:
    if col not in df.columns:
        raise ValueError(f'❌ Column not found: "{col}" — CSV mein columns hain: {df.columns.tolist()}')

# Normalize — strip whitespace
df[COL_NOTEBOOK]   = df[COL_NOTEBOOK].astype(str).str.strip()
df[COL_AUTOMATION] = df[COL_AUTOMATION].astype(str).str.strip()
df[COL_TRINO]      = df[COL_TRINO].astype(str).str.strip()

print(f'✅ CSV loaded: {len(df)} rows')

# ── Compare ──
df['NB == AUTO']    = df[COL_NOTEBOOK] == df[COL_AUTOMATION]
df['NB == TRINO']   = df[COL_NOTEBOOK] == df[COL_TRINO]
df['AUTO == TRINO'] = df[COL_AUTOMATION] == df[COL_TRINO]
df['ALL SAME']      = df['NB == AUTO'] & df['NB == TRINO'] & df['AUTO == TRINO']

# ── Summary ──
total = len(df)
nb_auto   = df['NB == AUTO'].sum()
nb_trino  = df['NB == TRINO'].sum()
auto_trino = df['AUTO == TRINO'].sum()
all_same  = df['ALL SAME'].sum()
mismatches = total - all_same

print()
print('=' * 60)
print('📊 RAW STRING COMPARISON SUMMARY')
print('=' * 60)
print(f'  Total rows          : {total}')
print(f'  Notebook == Auto    : {nb_auto}/{total}  {"✅" if nb_auto == total else "❌"}')
print(f'  Notebook == Trino   : {nb_trino}/{total}  {"✅" if nb_trino == total else "❌"}')
print(f'  Auto == Trino       : {auto_trino}/{total}  {"✅" if auto_trino == total else "❌"}')
print(f'  All 3 Same          : {all_same}/{total}  {"✅" if all_same == total else "❌"}')
print(f'  Mismatches          : {mismatches}')
print('=' * 60)

if mismatches == 0:
    print('\n✅ ALL ROWS MATCH — Teeno columns identical hain!')
else:
    print(f'\n❌ {mismatches} MISMATCH(ES) FOUND — Neeche detail dekho!')

---
## 🔎 Cell 4 — Mismatch Detail

In [ ]:
mismatch_df = df[~df['ALL SAME']].copy().reset_index()
mismatch_df.rename(columns={'index': 'original_row'}, inplace=True)
mismatch_df['original_row'] = mismatch_df['original_row'] + 1  # 1-based

if mismatch_df.empty:
    print('✅ Koi mismatch nahi — sab rows match hain!')
else:
    print(f'❌ Mismatch rows ({len(mismatch_df)}):\n')

    for _, row in mismatch_df.iterrows():
        print(f'Row {int(row["original_row"])}:')
        print(f'  NB == AUTO    : {row["NB == AUTO"]}')
        print(f'  NB == TRINO   : {row["NB == TRINO"]}')
        print(f'  AUTO == TRINO : {row["AUTO == TRINO"]}')

        # Find first char difference
        pairs = [
            ('Notebook vs Auto',  row[COL_NOTEBOOK], row[COL_AUTOMATION]),
            ('Notebook vs Trino', row[COL_NOTEBOOK], row[COL_TRINO]),
            ('Auto vs Trino',     row[COL_AUTOMATION], row[COL_TRINO]),
        ]
        for label, s1, s2 in pairs:
            if s1 != s2:
                # Find first diff
                first_diff = next((i for i, (a, b) in enumerate(zip(s1, s2)) if a != b),
                                   min(len(s1), len(s2)))
                print(f'  [{label}] first diff at index {first_diff}:')
                print(f'    s1: ...{repr(s1[max(0,first_diff-10):first_diff+20])}...')
                print(f'    s2: ...{repr(s2[max(0,first_diff-10):first_diff+20])}...')
                print(f'    Length: s1={len(s1)}, s2={len(s2)}')
        print()